# LOTR : The Lord of the RAGs


<font color='red'>**Important**: </font>
- **If the kernel doesn't attach automatically, or if you need to restart the kernel, please choose 'conda_python3' kernel.**
>* <font color='red'>Ensure that you are running this notebook on 'conda_python3' kernel: </font>
>* <font color='red'>you can switch kernels by clicking on the kernel name in the upper right corner of this notebook </font>
- Run each block of code one by one (starting from the top) either by clicking on "Run" on the menu above or by pressing SHIFT+ENTER on your keyboard.
- The blocks of code/ Notebook cells must be run sequentially, from top to bottom
- If your notebook hangs, restart the kernel by clicking on ``Kernel`` on the menu bar -> `` Restart kernel and clear outputs of all cells..`` 
- For the blocks that you need to run to complete the tasks, look for **"YOUR TURN"** in the title.
>* <font color='red'>ensure that you have run all the cells before comleting the sections marked with **"YOUR TURN"** </font>
- Don't alter the code in sections/cells with ``YOUR TURN`` in the title. Just run them.


## Introduction

You are working as a developer for a leading investment bank that has a global presence, your customers are have deep investments in the US capital markets and depend on the 10k forms (among several others) filed by publicly listed companies to keep a track of their investments.

The bank has already developed a front-end interface and has requested you to create an API-based serverless system to respond to customer queries. They have set up an API gateway that seamlessly integrates with their front-end, but they require assistance in setting up the GenerativeAI component.

In this challenge, you will first learn how to set up an Amazon Bedrock Knowledge Base. Next, you will learn how to retrieve data from Amazon Bedrock Knowledge Bases. After that, you will learn how to use prompt to query in multiple languages.


## Overview
Here is what you are going to do in this challenge:


   *[Task 1 : Enable Foundation Models](#enable-foundation-models)
   
   *[Task 2 : Know your Knowledge Base ID](#know-your-knowledge-base-id)
   
   *[Task 3 : Write a prompt](#write-a-prompt)
   
   *[Task 4 : Query translation](#query-translation)
   


# Install required libraries
the errors around various packages and libraries being incompatible can be safely ignored

In [ ]:
import boto3, pprint, json, os
import ipywidgets as widgets
from botocore.exceptions import ClientError

## Helper functions do not modify

In [ ]:
from botocore.client import Config

pp = pprint.PrettyPrinter(indent=2)
session = boto3.session.Session()
region = session.region_name

s3_client = boto3.client('s3')
bedrock_config = Config(connect_timeout=120, read_timeout=120, retries={'max_attempts': 0})

bedrock = boto3.client('bedrock', region)
bedrock_runtime = boto3.client('bedrock-runtime', region)

bedrock_agent_client = boto3.client('bedrock-agent', region)
bedrock_agent_runtime = boto3.client('bedrock-agent-runtime',
                                     region_name=region,
                                     config=bedrock_config
                                    )

s3_buckets = s3_client.list_buckets()
for bucket in s3_buckets['Buckets']:
    if 'lotrbucket' in bucket['Name'].lower():
        bucket_name = bucket['Name']
        break


def submit_your_answer(task_file_name, task_response):
    '''Upload your response to S3 in json format. This acts as your submission!
       Arguments:
           task_file_name (str): name of the task file you are submitting 
           task_response (Dictionary): Response dictionary 
    ''' 
    task_json = json.dumps(task_response)
    with open(task_file_name, 'w') as f:
        f.write(task_json)
    s3_client.upload_file(Bucket=bucket_name, Filename=os.path.join('./', task_file_name), Key=task_file_name)

def get_contexts(query, kbId, numberOfResults=10):
    # getting the contexts for the query from the knowledge base
    results = bedrock_agent_runtime.retrieve(
        retrievalQuery= {
            'text': query
        },
        knowledgeBaseId=kbId,
        retrievalConfiguration= {
            'vectorSearchConfiguration': {
                'numberOfResults': numberOfResults
            }
        }
    )
    #  creating a list to store the contexts
    contexts = []
    #   adding the contexts to the list
    for retrievedResult in results['retrievalResults']: 
        contexts.append(retrievedResult['content']['text'])
    #  returning the contexts
    return contexts

def answer_query_converse(user_input, model_id):
    # Use the Conversation API to send a text message to Nova Lite.
    user_message  = f"""
    You are a multilingual Question and answering assistant 
    Your responsibility is to answer user questions based on provided context
    Your answer must be in the same language as that of the question asked by the user within the <question> </question> tags
    
    For example :
    If the question in the <question> </question> tags is in German, your reply must be in German language
    If the question in the <question> </question> tags is in French, your reply must be in French language
    so on and so forth
    
    If you cannot identify the language say "I do not understand the language"

    from the context in the <context> </context> tags, use only the context that matches the language of the user's query in the <question> </question> tags 
    
    If you do not have any context within the <context> </context> tags say "I do not have enough context to answer the query"
    
    Here is the context to reference while answering the user's query:
    <context>
    {get_contexts(user_input, knowledge_base_id)}
    </context>

    Referencing the context, answer the user question
    <question>
    {user_input}
    </question>
    """
    conversation = [
        {
            "role": "user",
            "content": [{"text": user_message}],
        }
    ]
    
    try:
        # Send the message to the model, using a basic inference configuration.
        response = bedrock_runtime.converse(
            modelId=model_id,
            messages=conversation,
            inferenceConfig={"maxTokens": 512, "temperature": 0.1, "topP": 0.9},
        )
    
        # Extract and print the response text.
        response_text = response["output"]["message"]["content"][0]["text"]
        
        return response_text
    
    except (ClientError, Exception) as e:
        print(f"ERROR: Can't invoke '{model_id}'. Reason: {e}")
        exit(1)
    
    
def convert_text_to_embeddings(text, modelId = "amazon.titan-embed-text-v2:0"):
    modelId = "amazon.titan-embed-text-v2:0"  # 
    accept = "application/json"
    contentType = "application/json"
    
    sample_model_input={
        "inputText": text,
        "dimensions": 256,
        "normalize": True
    }
    
    body = json.dumps(sample_model_input)
    response = bedrock_runtime.invoke_model(body=body, 
                                            modelId=modelId, 
                                            accept=accept, 
                                            contentType=contentType)
    response_body = json.loads(response.get('body').read())
    embedding = response_body.get("embedding")
    return embedding

# Widget definitions
kb_id=widgets.Text(
    value='None',
    placeholder='Paste/Type in the KB Id here',
    description='Enter Knowledge Base ID:',
    disabled=False,
    style = {'description_width': 'initial'},
    layout=widgets.Layout(width='50%')
)

model  = widgets.Dropdown(
    options=[
             ('Nova Lite', 'amazon.nova-lite-v1:0')],
    value='None',
    description='Choose Nova Lite:',
    style = {'description_width': 'initial'},
    layout=widgets.Layout(width='50%')
)

query = widgets.Textarea(
    value='Type in your query here',
    placeholder='Type something',
    description='query :',
    disabled=False,
    style = {'description_width': 'initial'},
    layout=widgets.Layout(width='95%')
)

query_response = widgets.Textarea(
    value='Copy-paste the response you got from the RAG system here',
    placeholder='Type something',
    description='query_response:',
    disabled=False,
    style = {'description_width': 'initial'},
    layout=widgets.Layout(width='95%')
)

choose_target_language  = widgets.Dropdown(
    options=[('German', 'de'),
             ('French', 'fr'),
             ('Portuguese', 'pt-PT'),
             ('Spanish', 'es'),
             ('None', 'None')],
    value='None',
    description='Choose a target language...',
    style = {'description_width': 'initial'},
    layout=widgets.Layout(width='60%')
)

translated_query_response = widgets.Textarea(
    value='Copy-paste the NON-English response you got from the RAG system here',
    placeholder='Type something',
    description='Translated query response:',
    disabled=False,
    style = {'description_width': 'initial'},
    layout=widgets.Layout(width='95%')
)

### Test Nova Lite Model Access
Let's verify that you have enabled access to the Nova Lite model through Amazon Bedrock.

If you get an error message that reads along the lines of ``ERROR: Can't invoke 'amazon.nova-lite-v1:0'. Reason: An error occurred (AccessDeniedException)``

You know you haven't set up the model access yet...

So go ahead and enable model access through bedrock...

In [ ]:
# Test Nova Lite access
test_conversation = [
    {
        "role": "user",
        "content": [{"text": "Hello, can you respond in English?"}],
    }
]

try:
    response = bedrock_runtime.converse(
        modelId="amazon.nova-lite-v1:0",
        messages=test_conversation,
        inferenceConfig={"maxTokens": 100, "temperature": 0.1},
    )
    print("✅ Nova Lite access confirmed!")
    print("Response:", response["output"]["message"]["content"][0]["text"])
except Exception as e:
    print(f"❌ Nova Lite access error: {e}")
    print("Please enable Nova Lite model access in Amazon Bedrock console.")

### Test Amazon Titan Text Embeddings V2

In [ ]:
# Test Titan Embeddings access
try:
    prompt_data = "Test embedding generation"
    modelId = "amazon.titan-embed-text-v2:0"
    accept = "application/json"
    contentType = "application/json"

    sample_model_input={
        "inputText": prompt_data,
        "dimensions": 256,
        "normalize": True
    }

    body = json.dumps(sample_model_input)
    response = bedrock_runtime.invoke_model(body=body, modelId=modelId, accept=accept, contentType=contentType)

    response_body = json.loads(response.get('body').read())
    embedding = response_body.get("embedding")
    print(f"✅ Titan Embeddings access confirmed!")
    print(f"The embedding vector has {len(embedding)} values")
except Exception as e:
    print(f"❌ Titan Embeddings access error: {e}")
    print("Please enable Titan Text Embeddings V2 model access in Amazon Bedrock console.")

<a id='enable-foundation-models'> </a>
## <font color='17C822'> YOUR TURN: Task 1: Check if your models have been provisioned </font>

<b> Whooo...   
    You've made it this far, Run the following cell to mark the completion of all what you did to achieve the completion of task 1...    
    You have now laid the foundations to complete the  tasks that follow..  </b>

In [ ]:
submit_your_answer('task1.json', 'task1')

<a id='know-your-knowledge-base-id'> </a>
## <font color='17C822'> YOUR TURN: Task 2: Know your Knowledge Base ID </font>

<b> Update the knowledge base variable below `kb_id` with the knowledge base id that has been created for you as a part of this jam challenge

In [ ]:
# run this cell to generate the text widget
# enter the value of the kb_id in the text widget, it will be saved, run the next cell to verify
kb_id 

In [ ]:
knowledge_base_id = kb_id.value
assert knowledge_base_id != 'None', 'Please run the above cell and enter the Knowledge Base ID in the text widget'
knowledge_base_id

### <font color='Magenta'> **Submission** </font>  
Do not change the contents of this cell and that of the succeeding cell...   
Just run it as it is once you have completed the ``Task 2`` above.  
This is your submission for Task #2. You'll receive credit shortly if you answer is correct.

In [ ]:
# DO NOT CHANGE THE CONTENTS OF THIS CELL...
# CHANGES MADE HERE CAN CAUSE YOUR SUBMISSION TO FAIL...
submit_your_answer('task2.json', kb_id.value)

## Select Nova Lite Model

In [ ]:
# run this cell to generate the drop down widget
# select Nova Lite from the dropdown, it will be saved, run the next cell to verify
model 

In [ ]:
assert model.value != 'None', 'Please run the above cell and choose Nova Lite from the drop down widget'
model.value

## Sync KB with the S3 datasource

### Please complete the below steps before moving on to next cell

#### 1. Go to AWS console
#### 2. Search for bedrock in the search bar
#### 3. Click on the hamburger icon ont he upper left corner of to expand the left hand side menu 
#### 4. On the left hand side menu Under Builder tools, Click on Knowledged bases
#### 5. Click on the Knowledge Base link [there is going to be only one under the section ``Knowledge bases (1)``]
#### 6. Scroll down to reach the section "Data source" 
#### 7. Select the radio button to the left of the data source [there is only one]
#### 8. Click the 'sync' button
#### 9. Wait for ~ 5 mins for the sync to complete
#### 10. Check : After successful completion of sync the value under "Last sync time" should change from "-" (hyphen) to a time stamp

<br>

<img style="text-align:left"/>
  <img src="https://aws-jam-challenge-resources.s3.amazonaws.com/lord-of-the-ra-gs/KB_Sync.png" alt="drawing" width="1000" />
</p>


<a id='write-a-prompt'> </a>
## <font color='17C822'> YOUR TURN: Task 3: Write a prompt</font>

<b> Write a prompt below asking details about Octank financial's data centers  </b>   
for example : "Tell me about Octank Financial's data centers..."

In [ ]:
# run this cell to generate the text widget
# enter the prompt in the text widget, it will be saved, run the next 2 cells to verify
query 

In [ ]:
assert query.value != 'Type in your query here', 'Please run the above cell and enter the query/prompt asking about Octank financials data centers'
query.value

In [ ]:
pp.pprint(answer_query_converse(query.value, model.value))

### <font color='Magenta'> **Submission** </font>  
Do not change the contents of this cell and that of the succeeding cell...   
Just run it as it is once you have completed the ``Task 3`` dictionary above.  
This is your submission for Task #3. You'll receive credit shortly if you answer is correct.

In [ ]:
# run this cell to generate the text widget
# enter the response in the text widget, it will be saved, run the next cell to verify
query_response 

In [ ]:
assert query_response.value != 'Copy-paste the response you got from the RAG system here', 'Copy-paste the response you got from executing the pp.pprint(answer_query_converse(query.value, model.value))  in the above widget'

In [ ]:
# DO NOT CHANGE THE CONTENTS OF THIS CELL...
# CHANGES MADE HERE CAN CAUSE YOUR SUBMISSION TO FAIL..
submit_your_answer('task3.json', convert_text_to_embeddings (query_response.value))

<a id='query-translation'> </a>
## <font color='17C822'> YOUR TURN: Task 4: Query Translation</font>
### Choose a language you want the query to be translated to , from the doropdown below

In [ ]:
# run this cell to generate the drop down widget
# select the value in the drop down widget, it will be saved, run the next cell to verify
choose_target_language 

In [ ]:
assert choose_target_language.value != 'None', 'Please choose a vaue from the above drop down'

In [ ]:
translate = boto3.client(service_name='translate', region_name=region, use_ssl=True)
source_language = 'en'
target_language = choose_target_language.value
user_query = query.value

### Refer the documentation for [Amazon Translate]( https://docs.aws.amazon.com/translate/latest/dg/get-started-sdk.html#examples-python) and 
* complete the arguments translate_text function below, 
* using the input parameters defined in the cell above

In [ ]:
translation_result=translate.translate_text(Text=user_query,
                                            SourceLanguageCode=source_language,
                                            TargetLanguageCode=target_language)

In [ ]:
translation_result.get('TranslatedText')

In [ ]:
pp.pprint(answer_query_converse(translation_result.get('TranslatedText'), model.value))

### Copy paste the output from the above cell into the text widget/edit box that comes up after executing the cell below

In [ ]:
# run this cell to generate the text widget
# enter the response in the text widget, it will be saved, run the next cell to verify
translated_query_response 

In [ ]:
translated_query_response.value

### <font color='Magenta'> **Submission** </font>  
Do not change the contents of this cell and that of the succeeding cell...   
Just run it as it is once you have completed the ``Task 4`` above.  
This is your submission for Task #4. You'll receive credit shortly if you answer is correct.

In [ ]:
# DO NOT CHANGE THE CONTENTS OF THIS CELL...
# CHANGES MADE HERE CAN CAUSE YOUR SUBMISSION TO FAIL...
submit_your_answer('task4.json', convert_text_to_embeddings (translated_query_response.value))